# 01 Data cleaning

**Goal:** load the 9 Olist tables, check what one row means in each, fix problems, and join everything into one clean table of orders for the analysis.
**Data:** Olist Brazilian E-Commerce Public Dataset (Kaggle), 99,441 orders, September 2016 to October 2018.
**Output:** `orders_clean.parquet` (99,441 orders × 27 columns) and a cleaning log: every rule, the rows it affects, and the decision.

In [1]:
import pandas as pd, sklearn
print(pd.__version__, sklearn.__version__)

3.0.6 1.9.1


## Step 1. Load the 9 tables and check their sizes

In [2]:
from pathlib import Path
import pandas as pd

RAW = Path(r"..\data\raw")
tables = {f.stem: pd.read_csv(f) for f in sorted(RAW.glob("*.csv"))}

for name, df in tables.items():
    print(f"{name:45} {df.shape[0]:>8} rows  {df.shape[1]:>3} cols")

olist_customers_dataset                          99441 rows    5 cols
olist_geolocation_dataset                      1000163 rows    5 cols
olist_order_items_dataset                       112650 rows    7 cols
olist_order_payments_dataset                    103886 rows    5 cols
olist_order_reviews_dataset                      99224 rows    7 cols
olist_orders_dataset                             99441 rows    8 cols
olist_products_dataset                           32951 rows    9 cols
olist_sellers_dataset                             3095 rows    4 cols
product_category_name_translation                   71 rows    2 cols


**What the row counts show**
- Customers 99,441 = orders 99,441. That is no coincidence: `customer_id` is created per order. The real person is `customer_unique_id`, which matters in stage 2.
- Order items 112,650 > orders: some orders have more than one item, so items are summed per order before joining.
- Payments 103,886 > orders: some orders were paid in several parts (for example card plus voucher), so payments are also summed per order.
- Reviews 99,224 < orders: some orders have no review, and some may have two. Both are checked in step 7.
- Geolocation has 1,000,163 rows for about 19,000 zip prefixes, so it is full of duplicates. It is not needed for cleaning.
- The translation table has 71 rows: 71 product categories with Portuguese names.

## Step 2. Orders: missing dates and delivery times

In [3]:
orders = tables["olist_orders_dataset"]

In [4]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [6]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [7]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [8]:
# A. Delivered orders that are missing dates
delivered = orders[orders['order_status'] == 'delivered']
delivered.isna().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64

In [9]:
# B. Missing delivery date, broken down by status
orders[orders['order_delivered_customer_date'].isna()]['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

Of 2,965 orders with no customer delivery date, 2,957 were never delivered (shipped, canceled, unavailable, invoiced, processing, created, approved). That is expected. 8 are marked delivered but have no date: data errors, excluded from the delivery analysis. 6 canceled orders do have a delivery date; they are kept and flagged. The delivery and review analysis keeps delivered orders with a delivery date: 96,470 orders.

In [10]:
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])
orders[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [11]:
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
orders['is_late'] = orders['delay_days'] > 0

In [12]:
orders[['delivery_days','delay_days','is_late']].describe()

,delivery_days,delay_days
count,96476.000000,96476.000000
mean,12.094086,-11.876881
std,9.551746,10.183854
min,0.000000,-147.000000
25%,6.000000,-17.000000
50%,10.000000,-12.000000
75%,15.000000,-7.000000
max,209.000000,188.000000


In [13]:
orders['is_late'].mean()

np.float64(0.06571736004263835)

Typical delivery takes 10 days, and most orders arrive well before the promised date (median 12 days early). Late orders are a minority, so the question for EDA is whether that minority drives the bad reviews.

In [14]:
(orders['delivery_days'] == 0).sum()

np.int64(13)

In [15]:
(orders['delivery_days'] > 60).sum()

np.int64(288)

In [16]:
has_date = orders['order_delivered_customer_date'].notna()
orders.loc[has_date, 'is_late'].mean()

np.float64(0.06773705377503213)

**Cleaning log: orders**

| Rule | Rows affected | Decision |
| --- | --- | --- |
| Not delivered (shipped, canceled, etc.), no delivery date | 2,957 | Exclude from delivery and review analysis |
| Marked delivered but no delivery date | 8 | Exclude, data error |
| Canceled but has a delivery date | 6 | Keep and flag |
| Delivered the same day | 13 | Keep |
| Delivery over 60 days | 288 | Keep, use medians |
| Late (delivered after the estimate) | 6,535 (6.8% of orders with a delivery date) | New column `is_late` |

## Step 3. Order items: one row per order

In [17]:
items = tables['olist_order_items_dataset']

In [18]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [19]:
items_per_order = items.groupby('order_id').agg(
    n_items=('order_item_id', 'count'),
    n_sellers=('seller_id', 'nunique'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
).reset_index()

In [20]:
items_per_order['freight_share'] = items_per_order['total_freight']/(items_per_order['total_price'] + items_per_order['total_freight'])

In [21]:
items_per_order.shape

(98666, 6)

In [22]:
items_per_order.describe()

,n_items,n_sellers,total_price,total_freight,freight_share
count,98666.000000,98666.000000,98666.000000,98666.000000,98666.000000
mean,1.141731,1.013622,137.754076,22.823562,0.208804
std,0.538452,0.122297,210.645145,21.650909,0.125749
min,1.000000,1.000000,0.850000,0.000000,0.000000
25%,1.000000,1.000000,45.900000,13.850000,0.116502
50%,1.000000,1.000000,86.900000,17.170000,0.183256
75%,1.000000,1.000000,149.900000,24.040000,0.275463
max,21.000000,5.000000,13440.000000,1794.960000,0.955451


Most orders are a single item at around 87 BRL, and shipping makes up about a fifth of the price. Expensive freight is a candidate driver of bad reviews: test it in EDA.

## Step 4. Payments: one row per order

In [23]:
payments = tables['olist_order_payments_dataset']
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [24]:
payments_per_order = payments.groupby('order_id').agg(
    n_payments = ('payment_sequential','count'),
    total_paid = ('payment_value' , 'sum'),
    max_installments = ('payment_installments','max'),
).reset_index()
payments_per_order.describe()

,n_payments,total_paid,max_installments
count,99440.000000,99440.000000,99440.000000
mean,1.044710,160.990267,2.930521
std,0.381166,221.951257,2.715685
min,1.000000,0.000000,0.000000
25%,1.000000,62.010000,1.000000
50%,1.000000,105.290000,2.000000
75%,1.000000,176.970000,4.000000
max,29.000000,13664.080000,24.000000


In [25]:
payments['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

**Payments per order.** 99,440 orders have a payment record, one less than the 99,441 orders, so exactly one order has no payment record (added to the cleaning log).

- `n_payments`: almost every order is paid in one go. The maximum of 29 is one order split into 29 payment rows, most likely vouchers.
- `total_paid`: median 105.29 BRL, higher than the items' median price of 86.90, because the paid amount includes shipping. The minimum of 0 is an order paid fully with a voucher or similar.

**Check: does what customers paid match price plus freight?** Items and payments are joined on `order_id`:

In [26]:
check = items_per_order.merge(payments_per_order,
                              on='order_id',
                              how='inner')
check['diff'] = check['total_paid'] - (check['total_price'] + check['total_freight'])
check['diff'].head()

0    0.000000e+00
1    0.000000e+00
2    0.000000e+00
3    0.000000e+00
4   -2.842171e-14
Name: diff, dtype: float64

In [27]:
check['diff'].describe()

count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: diff, dtype: float64

In [28]:
(check['diff'].abs() > 1).sum()

np.int64(249)

Paid amounts match item price plus freight for 99.7% of orders, so the price and freight data can be trusted. 249 orders differ by more than 1 BRL (from −52 to +183), likely due to vouchers and instalment interest. They are kept, and item price and freight are used for the analysis.

| Rule | Rows affected | Decision |
| --- | --- | --- |
| Paid amount differs from price + freight by more than 1 BRL | 249 | Keep, use item values |

## Step 5. Products and categories

In [29]:
products = tables['olist_products_dataset']

In [30]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [31]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [32]:
translation = tables['product_category_name_translation']

In [33]:
translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [34]:
translation.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

**A. Are the 610 products with no category missing their other details too?**

In [35]:
no_cat = products['product_category_name'].isna()
products.loc[no_cat].isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

**B. Fix the misspelled column names** (`lenght` → `length`).

In [36]:
products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length',
})

**C. Which categories have no English translation?**

In [37]:
in_products = set(products['product_category_name'].dropna())
in_translation = set(translation['product_category_name'])
in_products - in_translation

{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}

**D. Add English names, then fill the gaps.**

In [38]:
products = products.merge(translation,
                          on='product_category_name',
                          how='left')
products['category'] = products['product_category_name_english'].fillna('unknown')
products['category'].value_counts().head(10)

category
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64

Two categories have no English name, so they are translated by hand:

In [39]:
manual = {
    'pc_gamer': 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos': 'small_kitchen_appliances',
}
for pt, en in manual.items():
    products.loc[products['product_category_name'] == pt, 'category'] = en

In [40]:
(products['category'] == 'unknown').sum()

np.int64(610)

In [41]:
products.loc[products['product_category_name'].isin(manual), 'category'].value_counts()

category
small_kitchen_appliances    10
pc_gamer                     3
Name: count, dtype: int64

**Result.** 610 products remain `unknown`: only the truly empty listings. 10 products are now `small_kitchen_appliances` and 3 are `pc_gamer`.

## Step 6. Customers: orders vs real people
Short, but it matters for stage 2: `customer_id` is created per order, `customer_unique_id` identifies the person.

In [44]:
customers = tables['olist_customers_dataset']

In [45]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [46]:
print(customers['customer_id'].nunique())
print(customers['customer_unique_id'].nunique())

99441
96096


In [47]:
orders_per_person = customers['customer_unique_id'].value_counts()
(orders_per_person > 1).sum()

np.int64(2997)

In [49]:
n_orders = customers['customer_id'].nunique()
n_people = customers['customer_unique_id'].nunique()
print(n_orders, n_people)

99441 96096


In [51]:
orders_per_person = customers['customer_unique_id'].value_counts()
n_repeat = (orders_per_person > 1).sum()

In [53]:
share_repeat = n_repeat / n_people
print(n_repeat, round(share_repeat * 100, 1), '%')

2997 3.1 %


99,441 orders came from 96,096 unique customers. Only 2,997 customers (3.1%) ordered more than once, so about 97% are one-time buyers. The marketplace depends almost entirely on acquiring new customers, which is expensive. Retention is the biggest untapped lever: post-purchase follow-ups, win-back campaigns and fixing bad delivery experiences. Stage 2 segments customers to target this.

## Step 7. Reviews: comments and duplicates

In [54]:
reviews = tables['olist_order_reviews_dataset']
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [55]:
reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [58]:
print(reviews['review_id'].nunique(), reviews['order_id'].nunique(), len(reviews))

98410 98673 99224


**Comments.** 58,247 of 99,224 reviews (59%) have no message, so about 41,000 reviews (41%) include written text: enough for NLP in stage 4.
Titles are rarer still: 88% have no title, so the message is the column that matters.
Every review has a score (none missing), so the score is the reliable signal.

**Duplicates.** Three counts: 98,410 unique review IDs, 98,673 unique orders, 99,224 rows.
- 99,224 rows vs 98,673 orders: 551 extra rows, because some orders have more than one review.
- 99,224 rows vs 98,410 review IDs: 814 extra rows, because some reviews are attached to more than one order, likely one review covering several orders bought together.

In [59]:
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

In [62]:
reviews_one = (reviews.sort_values('review_answer_timestamp').drop_duplicates(subset='order_id', keep='last'))
print(len(reviews_one), reviews_one['order_id'].nunique())

98673 98673


In [65]:
reviews_one['has_comment'] = reviews_one['review_comment_message'].notna()
reviews_one['comment_length'] = reviews_one['review_comment_message'].str.len()
reviews_one.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_comment,comment_length
37547,6916ca4502d6d3bfd39818759d55d536,bfbd0f9bdef84302105ad712db648a6c,1,NaN,nao recebi o produto e nem resposta da empresa,2016-10-06 00:00:00,2016-10-07 18:32:28,True,46.0
5503,49f695dffa457eaba90d388a5c37e942,e5215415bb6f76fe3b7cb68103a0d1c0,1,NaN,"PRODUTO NÃO CHEGOU,E JÁ PASSOU O PRAZO DE ENTREGA",2016-10-09 00:00:00,2016-10-11 14:31:29,True,49.0
60439,743d98b1a4782f0646898fc915ef002a,e2144124f98f3bf46939bc5183104041,4,NaN,NaN,2016-10-15 00:00:00,2016-10-16 03:20:17,False,NaN
28075,53752edb26544dd41c1209f582c9c589,b8b9d7046c083150cb5360b83a8ebb51,5,NaN,O pedido foi entregue antes do prazo pr0metido,2016-10-16 01:00:00,2016-10-16 15:45:11,True,46.0
41042,b2d5d8db2a841d27a72e4c06c6212368,9aa3197e4887919fde0307fc23601d7a,4,NaN,Só chegou uma parte do pedido ate agora..,2016-10-15 00:00:00,2016-10-17 21:02:49,True,41.0


**Cleaning log: reviews**

| Rule | Rows affected | Decision |
| --- | --- | --- |
| Orders with more than one review | 551 | Keep the latest per order |

## Step 8. Join everything into one table and save it

In [66]:
first_item = items[items['order_item_id'] == 1][['order_id', 'product_id', 'seller_id']]
first_item = first_item.merge(products[['product_id', 'category']], on='product_id', how='left')
first_item.head()

,order_id,product_id,seller_id,category
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,garden_tools


In [67]:
orders_clean = (orders
    .merge(items_per_order, on='order_id', how='left')
    .merge(payments_per_order, on='order_id', how='left')
    .merge(first_item, on='order_id', how='left')
    .merge(customers[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id', how='left')
    .merge(reviews_one[['order_id', 'review_score', 'has_comment', 'comment_length']], on='order_id', how='left')
)
print(orders_clean.shape, orders_clean['order_id'].nunique())

(99441, 27) 99441


In [68]:
orders_clean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delay_days,...,total_paid,max_installments,product_id,seller_id,category,customer_unique_id,customer_state,review_score,has_comment,comment_length
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,...,38.71,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,housewares,7c396fd4830fd04220f754e42b4e5bff,SP,4.0,True,170.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,...,141.46,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,perfumery,af07308b275d755c9edb36a90c618231,BA,4.0,True,20.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,...,179.12,3.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,auto,3a653a41f6f9fc3d2a113cf8398680e8,GO,5.0,False,NaN
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,...,72.20,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,pet_shop,7c142cf63193a1473d2e66489a9ae977,RN,5.0,True,105.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,...,28.62,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,stationery,72632f0f9dd73dfee390c9b22eb56dd6,SP,5.0,False,NaN


In [69]:
orders_clean.isna().sum().sort_values(ascending=False).head(10)

comment_length                   58666
delivery_days                     2965
delay_days                        2965
order_delivered_customer_date     2965
order_delivered_carrier_date      1783
category                           775
product_id                         775
n_sellers                          775
total_price                        775
n_items                            775
dtype: int64

In [70]:
orders_clean['review_score'].value_counts().sort_index()

review_score
1.0    11363
2.0     3131
3.0     8133
4.0    19038
5.0    57008
Name: count, dtype: int64

**Check after joining.** 2,965 orders without a delivery date and 775 without items, exactly as found earlier, so the joins behaved.
`comment_length` is missing for 58,666 orders: those whose review has no message, plus the 768 orders with no review.
Review scores: 57,008 five-star (58%) and 11,363 one-star (12%). With 3,131 two-star, about 15% are bad reviews: the target for the stage 3 model.

In [71]:
from pathlib import Path
PROCESSED = Path(r"..\data\processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

orders_clean.to_parquet(PROCESSED / "orders_clean.parquet", index=False)
reviews_one.to_parquet(PROCESSED / "reviews_clean.parquet", index=False)
print(sorted(p.name for p in PROCESSED.iterdir()))

['orders_clean.parquet', 'reviews_clean.parquet']


## Summary

| Question | Answer |
| --- | --- |
| What is one row in each table? | Orders: one order. Items: one item in an order. Payments: one payment part. Reviews: one review. Customers: one order's customer. |
| Why are dates missing? | 2,957 orders were never delivered, as expected. 8 were "delivered" with no date: data errors. |
| How long does delivery take? | Median 10 days. Most orders arrive about 12 days before the promised date. |
| How many orders are late? | About 6.8% of delivered orders. |
| What does a typical order look like? | 1 item, about 87 BRL, with shipping around 18% of the price. |
| Do the payments match the prices? | Yes, for 99.7% of orders. 249 differ, likely vouchers and interest. |
| Are all categories translated? | 2 were not, and were translated by hand. 610 products have no category. |
| How many real customers come back? | 96,096 people, of whom only 3.1% ordered twice. |
| Is there one review per order? | No: 551 extra rows. The latest review per order is kept. |
| How satisfied are customers? | 58% give 5 stars, about 15% give 1 to 2 stars. |

**What was done**
- Loaded and profiled 9 tables.
- Converted dates and built 3 features: `delivery_days`, `delay_days` and `is_late`.
- Summarised items and payments to one row per order, and checked them against each other.
- Fixed column names and categories.
- Removed duplicate reviews.
- Joined everything into one clean table (99,441 orders × 27 columns) and saved it as Parquet.
- Kept a cleaning log: every rule, with its row count and decision.